In [1]:
import h5py
import matplotlib.pyplot as plt
import matplotlib.markers as mrkrs
import numpy as np
import pandas as pd
import os
from tqdm import tqdm

In [2]:
in_data = "./DATASET"
out_data = "./csvs2/"
scan_imgs_dir = "./scan_imgs2/"
os.makedirs(out_data, exist_ok=True)
os.makedirs(scan_imgs_dir, exist_ok=True)

## Current Threshold: redefining/restricting operational zone
c_thresh = 4300

## Thresholding distance to preceeding point to detemrine continuous scan
delta_thresh = .1

## Looping through each layer and creating preprocessed csvs
k = 0
all_deltas = None
for f in tqdm(os.listdir(in_data)[k:]):
    in_path = os.path.join(in_data, f)
    if os.path.exists(in_path) and os.path.basename(in_path).endswith('.hdf5'):
        tag = f.split(".")[0]
        
        out_path = os.path.join(out_data, tag + ".csv")
        img_path = os.path.join(scan_imgs_dir, tag+".png")
        
        file_info = h5py.File(in_path, 'r')
        file_data = file_info['OpenData']
        x = file_data[0]
        y = file_data[1]
        power = file_data[2]
        speed = file_data[3]
        dia = file_data[4]
        cur = file_data[5]
        sig = file_data[6]
        id1 = file_data[7]
        id2 = file_data[8]
        c1 = file_data[9]
        c2 = file_data[10]

        ## N x 11 datapoints per layer
        data = np.vstack([x, y, power, speed, dia, cur, sig, id1, id2, c1, c2]).T

        ## Filter out where the current is in operational zone
        turn_around_condition = data[:, 5] <= c_thresh
        turn_around_data = data[turn_around_condition]

        ## Determine continuous scans and grouping with scan numbers
        x = turn_around_data[:, 0]
        y = turn_around_data[:, 1]
        cur = turn_around_data[:, 5]
        deltas = np.zeros_like(x)
        scan_nums = np.zeros_like(x, dtype=int)
        scan = int(0)
        
        for i in range(1, len(deltas)):
            deltas[i] = np.sqrt((x[i]-x[i-1])**2 + (y[i]-y[i-1])**2)
            if deltas[i] > delta_thresh:
                scan+= int(1)
            scan_nums[i] = scan 

        ## Expanding data array to include euclidean distance from previous point and scan number
        deltas = np.expand_dims(deltas, -1)
        scan_nums_col = np.expand_dims(scan_nums, -1).astype(int)
        turn_around_data = np.hstack([turn_around_data, deltas, scan_nums_col])


        # if isinstance(all_deltas, type(None)):
        #     all_deltas = deltas
        #     # print(all_deltas.shape)
        # else:
        #     all_deltas = np.vstack([all_deltas, deltas])

        # break

        ## Save to CSV
        headers = [
            "x", "y", "power", "speed", "spot_diameter", "laser_current", 
            "signal", "ID1", "ID2", "C1", "C2", "delta", "scan_number"]
        df = pd.DataFrame(turn_around_data, columns=headers)
        df.to_csv(out_path)
        
        ## Temporary: Visualize segments
        hues = ["red", "blue", "green", "purple", "orange", "black", "yellow", "brown"]
        coloring = []
        for i in range(len(x)):
            col_idx = int(scan_nums[i]) % len(hues)
            coloring.append(hues[col_idx])
    
        j=len(x)
        plt.figure()
        sc = plt.scatter(data[:, 0], data[:, 1], c="grey", marker=mrkrs.MarkerStyle("o"), s=0.5, alpha=0.1, edgecolor='none')
        plt.scatter(data[0, 0], data[0, 1], color="black", marker='s', s=5)
        plt.scatter(data[-1, 0], data[-1, 1], color="black", marker='+', s=5)
        plt.scatter(x[:j], y[:j], c=coloring[:j], s =2)
        plt.savefig(img_path, dpi=600)
        plt.close()
        
        
        # break
        # plt.savefig("/home/kjw/Documents/sample.png", dpi=600)
        # plt.close()

        # plt.figure()
        # plt.plot(data[:, 0], data[:, 1], ':y', linewidth=1)
        # plt.scatter(data[0, 0], data[0, 1], color="black", marker='s', s=5)
        # plt.scatter(data[-1, 0], data[-1, 1], color="black", marker='o', s=5)
        # plt.scatter(x[:j], y[:j], c=coloring[:j], s =2)
        # break

        
        # plt.savefig("/home/kjw/Documents/sample2.png", dpi=600)
        # plt.close()

        # plt.figure()
        # sc = plt.scatter(data[:, 0], data[:, 1], c=data[:, 5], s =1)
        # plt.scatter(data[0, 0], data[0, 1], color="black", marker='s', s=5)
        # plt.scatter(data[-1, 0], data[-1, 1], color="black", marker='+', s=5)
        # plt.colorbar(sc)
        # plt.savefig(img_path, dpi=600)
        # plt.close()
        
        # break


        # seg_num, seg_start_idx = np.unique(turn_around_data[:, 12], return_index=True)
        # print("seg_num")
        # print(seg_num)

        # print("seg_start_idx")
        # print(seg_start_idx)
        
        # # for i in seg_num:
        # start_xs = x[seg_start_idx]
        # start_ys = y[seg_start_idx]
        # plt.scatter(start_xs, start_ys, color="green", marker='s', s=2)
        # break
        

        

100%|█████████████████████████████████| 381/381 [05:51<00:00,  1.09it/s]


In [3]:
# all_deltas.shape

In [4]:
# thresh = 0.6
# b =1
# span = 10000
# subset = all_deltas[span*(b-1):span*b]
# subset = subset[subset<thresh]
# plt.hist(subset, bins=20)